# 02 - Thí nghiệm Khảo sát & Cắt bỏ thành phần (Ablation Studies)

Phụ trách: **Người 1**. Notebook này **chỉ đọc log thô** từ `results/logs/ablations/` do `python -m eval.ablations` sinh ra (không nhập tay số liệu) để tạo bảng và hình cho Chương 4 của báo cáo.

### Ba thí nghiệm Ablation trọng tâm:
1. **Ablation 1 (`alphabeta_pruning`)**: So sánh `alphabeta` (có cắt tỉa Alpha-Beta) vs `alphabeta_nopruning` (Minimax thuần tắt cắt tỉa ở cùng `depth=4`) — đánh giá mức giảm số nút duyệt (`nodes`), độ trễ (`ms/move`) và hiệu quả dưới giới hạn thời gian.
2. **Ablation 2 (`mcts_simulations`)**: Khảo sát `mcts_100`, `mcts_500`, `mcts` (`1000`), `mcts_5000` — đánh giá sự tăng trưởng sức mạnh và chi phí thời gian theo ngân sách mô phỏng.
3. **Ablation 3 (`heuristic_components`)**: Trả lời *Câu hỏi nghiên cứu 2* — so sánh hàm đánh giá đầy đủ (Góc + Mobility + X/C-square + Số quân) với biến thể chỉ đếm quân (`alphabeta_discs_only` và `heuristic_discs_only`).

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
from eval.metrics import wilson_interval

LOGS = Path(os.environ.get("LOGS_DIR", REPO / "results" / "logs"))
FIGS = Path(os.environ.get("FIG_DIR", REPO / "results" / "figures"))
FIGS.mkdir(parents=True, exist_ok=True)

ABL_DIR = LOGS / "ablations"
EXPERIMENTS = ["alphabeta_pruning", "mcts_simulations", "heuristic_components"]
summaries = {}
for exp in EXPERIMENTS:
    p = ABL_DIR / exp / "summary.json"
    with open(p, encoding="utf-8") as f:
        summaries[exp] = json.load(f)
    print(f"Loaded {exp:22s}: {summaries[exp]['meta']['total_games']:4d} games | elapsed={summaries[exp]['meta']['elapsed_seconds']}s")

## 1. Ablation 1: Alpha-Beta Bật vs Tắt cắt tỉa (`alphabeta` vs `alphabeta_nopruning`)

- Ở chế độ **`default_config` (`time_budget = null`, cùng `depth = 4`)**: Kiểm chứng lý thuyết rằng cắt tỉa Alpha-Beta giữ nguyên chất lượng nước đi (điểm đối đầu cân bằng ~0.50) nhưng giảm mạnh số nút duyệt (`nodes`) và thời gian suy nghĩ.
- Ở chế độ **`time_per_move` (`time_budget = 0.5s`)**: Kiểm tra xem việc tắt cắt tỉa có làm Minimax bị lố ngân sách thời gian (`over_budget`) hay bị cắt sớm không.

In [ ]:
s_ab = summaries["alphabeta_pruning"]
ab_rows = []
for mode_name, mode_data in s_ab["modes"].items():
    m = mode_data["matches"][0]
    for ag, is_a in (("alphabeta", True), ("alphabeta_nopruning", False)):
        sc = m["a_summary"]["score"] if is_a else 1.0 - m["a_summary"]["score"]
        ci = m["a_score_bootstrap_ci"] if is_a else [1.0 - m["a_score_bootstrap_ci"][1], 1.0 - m["a_score_bootstrap_ci"][0]]
        lat = m["latency"][ag]
        nodes = m["cost"][ag]["nodes_mean"]
        over = mode_data["budget"]["by_agent"][ag]["over_budget_moves"]
        ab_rows.append({
            "Mode": mode_name,
            "Agent": ag,
            "Score": round(sc, 4),
            "Bootstrap 95% CI": f"[{ci[0]:.3f}, {ci[1]:.3f}]",
            "Avg Nodes / Move": round(nodes, 1),
            "Mean Latency (ms)": round(lat["mean_ms"], 2),
            "p95 Latency (ms)": round(lat["p95_ms"], 2),
            "Over-budget Moves": over,
        })

df_ab = pd.DataFrame(ab_rows)
display(df_ab)

# Tính tỷ lệ tăng tốc ở chế độ default_config (cùng đi hết depth=4)
def_ab = df_ab[df_ab["Mode"] == "default_config"].set_index("Agent")
node_ratio = def_ab.loc["alphabeta_nopruning", "Avg Nodes / Move"] / def_ab.loc["alphabeta", "Avg Nodes / Move"]
time_ratio = def_ab.loc["alphabeta_nopruning", "Mean Latency (ms)"] / def_ab.loc["alphabeta", "Mean Latency (ms)"]
print(f"=> Ở cùng depth=4 (default_config): Cắt tỉa Alpha-Beta giảm {node_ratio:.2f}x số nút duyệt "
      f"({def_ab.loc['alphabeta_nopruning', 'Avg Nodes / Move']:.0f} -> {def_ab.loc['alphabeta', 'Avg Nodes / Move']:.0f} nodes/nước) "
      f"và nhanh hơn {time_ratio:.2f}x ({def_ab.loc['alphabeta_nopruning', 'Mean Latency (ms)']:.2f}ms -> {def_ab.loc['alphabeta', 'Mean Latency (ms)']:.2f}ms).")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6), dpi=140)
modes_list = ["default_config", "time_per_move"]
x = np.arange(len(modes_list))
w = 0.34

for i, (ag, label, col) in enumerate([
    ("alphabeta", "Alpha-Beta (pruning=True)", "#2b6cb0"),
    ("alphabeta_nopruning", "Minimax thuần (pruning=False)", "#e53e3e"),
]):
    sub = df_ab[df_ab["Agent"] == ag].set_index("Mode").loc[modes_list]
    b1 = ax1.bar(x + (i - 0.5)*w, sub["Avg Nodes / Move"], w, label=label, color=col, edgecolor="black", linewidth=0.6)
    b2 = ax2.bar(x + (i - 0.5)*w, sub["Mean Latency (ms)"], w, label=label, color=col, edgecolor="black", linewidth=0.6)
    for b in b1:
        ax1.text(b.get_x() + b.get_width()/2, b.get_height() + 180, f"{b.get_height():,.0f}", ha="center", fontsize=9, fontweight="bold")
    for b in b2:
        ax2.text(b.get_x() + b.get_width()/2, b.get_height() + 4, f"{b.get_height():.1f} ms", ha="center", fontsize=9, fontweight="bold")

ax1.set_xticks(x, ["default_config (Độ sâu 4 đầy đủ)", "time_per_move (Giới hạn 0.5s)"])
ax1.set_ylabel("Số nút duyệt trung bình / nước (Nodes)")
ax1.set_title("Ablation 1: Số nút duyệt khi Bật vs Tắt cắt tỉa", fontweight="bold")
ax1.set_ylim(0, max(df_ab["Avg Nodes / Move"]) * 1.18)
ax1.legend()
ax1.grid(axis="y", linestyle=":", alpha=0.5)

ax2.set_xticks(x, ["default_config (Độ sâu 4 đầy đủ)", "time_per_move (Giới hạn 0.5s)"])
ax2.set_ylabel("Thời gian suy nghĩ trung bình / nước (ms)")
ax2.set_title("Ablation 1: Thời gian mỗi nước khi Bật vs Tắt cắt tỉa", fontweight="bold")
ax2.set_ylim(0, max(df_ab["Mean Latency (ms)"]) * 1.18)
ax2.legend()
ax2.grid(axis="y", linestyle=":", alpha=0.5)

fig.tight_layout()
fig.savefig(FIGS / "02_ablation_alphabeta_pruning.png", bbox_inches="tight")
plt.show()
print("Đã lưu hình:", FIGS / "02_ablation_alphabeta_pruning.png")

## 2. Ablation 2: Ảnh hưởng của số mô phỏng MCTS (`100`, `500`, `1000`, `5000`)

- Ở chế độ **`default_config` (`time_budget = null`)**: Mỗi biến thể chạy đúng số mô phỏng cấu hình (`100`, `500`, `1000`, `5000`). Đây là phép đo trực tiếp xem tăng `simulations` giúp MCTS mạnh lên bao nhiêu và tốn thêm bao nhiêu thời gian.
- Ở chế độ **`time_per_move` (`time_budget = 0.5s`)**: Mọi biến thể MCTS đều chạy theo đồng hồ 0.5s/nước nên đạt số mô phỏng thực tế xấp xỉ nhau (~460–470 sims/nước) và điểm số hội tụ về ~0.50 (kiểm chứng tính nhất quán của cơ chế dừng theo giờ).

In [ ]:
s_mcts = summaries["mcts_simulations"]
mcts_agents = s_mcts["config"]["agents"]
sim_budgets = {"mcts_100": 100, "mcts_500": 500, "mcts": 1000, "mcts_5000": 5000}

mcts_rows = []
for mode_name, mode_data in s_mcts["modes"].items():
    for ag in mcts_agents:
        w, d, l, g = 0, 0, 0, 0
        ms_list, p95_list, sims_list = [], [], []
        for m in mode_data["matches"]:
            if m["a"] == ag or m["b"] == ag:
                is_a = (m["a"] == ag)
                s = m["a_summary"]
                w += s["wins"] if is_a else s["losses"]
                d += s["draws"]
                l += s["losses"] if is_a else s["wins"]
                g += s["games"]
                ms_list.append(m["latency"][ag]["mean_ms"])
                p95_list.append(m["latency"][ag]["p95_ms"])
                sims_list.append(m["cost"][ag]["simulations_mean"])
        lo, hi = wilson_interval(w + 0.5 * d, g)
        mcts_rows.append({
            "Mode": mode_name,
            "Agent": ag,
            "Configured Sims": sim_budgets[ag],
            "Actual Sims / Move": round(float(np.mean(sims_list)), 1),
            "W - D - L": f"{w} - {d} - {l}",
            "Mean Score": round(mode_data["mean_score"][ag], 4),
            "95% Wilson CI": f"[{lo:.3f}, {hi:.3f}]",
            "ci_lo": lo,
            "ci_hi": hi,
            "Mean Latency (ms)": round(float(np.mean(ms_list)), 2),
            "p95 Latency (ms)": round(float(np.max(p95_list)), 2),
        })

df_mcts = pd.DataFrame(mcts_rows)
print("=== KẾT QUẢ ABLATION 2: SỐ MÔ PHỎNG MCTS ===")
display(df_mcts.drop(columns=["ci_lo", "ci_hi"]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.8, 4.8), dpi=140)
sub_def = df_mcts[df_mcts["Mode"] == "default_config"].sort_values("Configured Sims")
sub_tpm = df_mcts[df_mcts["Mode"] == "time_per_move"].sort_values("Configured Sims")

sims_x = sub_def["Configured Sims"].values
sc_def = sub_def["Mean Score"].values
ax1.errorbar(
    sims_x, sc_def,
    yerr=[sc_def - sub_def["ci_lo"].values, sub_def["ci_hi"].values - sc_def],
    marker="o", lw=2.2, capsize=5, color="#2b6cb0", label="default_config (Chạy đúng số mô phỏng)",
)
sc_tpm = sub_tpm["Mean Score"].values
ax1.errorbar(
    sims_x, sc_tpm,
    yerr=[sc_tpm - sub_tpm["ci_lo"].values, sub_tpm["ci_hi"].values - sc_tpm],
    marker="s", lw=1.8, linestyle="--", capsize=4, color="#718096", label="time_per_move (Cố định 0.5s/nước)",
)
for x_v, y_v in zip(sims_x, sc_def):
    ax1.annotate(f"{y_v:.3f}", (x_v, y_v), textcoords="offset points", xytext=(0, 9), ha="center", fontsize=9, fontweight="bold")

ax1.set_xscale("log")
ax1.set_xticks(sims_x, [str(s) for s in sims_x])
ax1.set_ylim(0.05, 0.98)
ax1.set_xlabel("Số mô phỏng cấu hình (simulations - thang log)")
ax1.set_ylabel("Mean Score (trung bình vòng tròn)")
ax1.set_title("Ablation 2: Sức mạnh MCTS theo số lần mô phỏng", fontweight="bold")
ax1.legend(fontsize=8.5)
ax1.grid(True, linestyle=":", alpha=0.6)

# Biểu đồ thời gian suy nghĩ theo số mô phỏng (default_config)
ax2.plot(sims_x, sub_def["Mean Latency (ms)"], marker="o", lw=2.2, color="#dd6b20", label="Mean Latency (ms/nước)")
ax2.plot(sims_x, sub_def["p95 Latency (ms)"], marker="^", lw=2.0, linestyle="--", color="#e53e3e", label="p95 Latency (ms/nước)")
for x_v, y_v in zip(sims_x, sub_def["Mean Latency (ms)"]):
    ax2.annotate(f"{y_v:.0f} ms", (x_v, y_v), textcoords="offset points", xytext=(4, 6), fontsize=8.5, fontweight="bold")
ax2.set_xscale("log")
ax2.set_yscale("log")
ax2.set_xticks(sims_x, [str(s) for s in sims_x])
ax2.set_xlabel("Số mô phỏng cấu hình (simulations - thang log)")
ax2.set_ylabel("Thời gian suy nghĩ mỗi nước (ms - thang log)")
ax2.set_title("Ablation 2: Chi phí thời gian theo số mô phỏng MCTS", fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(True, linestyle=":", alpha=0.6)

fig.tight_layout()
fig.savefig(FIGS / "02_ablation_mcts_simulations.png", bbox_inches="tight")
plt.show()
print("Đã lưu hình:", FIGS / "02_ablation_mcts_simulations.png")

## 3. Ablation 3: Thành phần hàm đánh giá — Trả lời Câu hỏi nghiên cứu 2

*Câu hỏi nghiên cứu 2: Hàm đánh giá có góc (Corners), tránh ô X/C và độ cơ động (Mobility) có mạnh hơn hàm chỉ đếm quân (`discs_only`) không, và mạnh hơn bao nhiêu?*

In [ ]:
s_heur = summaries["heuristic_components"]
h_agents = s_heur["config"]["agents"]
H_LABELS = {
    "alphabeta": "Alpha-Beta (Full Eval)",
    "alphabeta_discs_only": "Alpha-Beta (Discs Only)",
    "heuristic": "Heuristic (Full Rules)",
    "heuristic_discs_only": "Heuristic (Discs Only)",
}

h_pair_rows = []
for mode_name, mode_data in s_heur["modes"].items():
    for m in mode_data["matches"]:
        s = m["a_summary"]
        ci = m["a_score_bootstrap_ci"]
        h_pair_rows.append({
            "Mode": mode_name,
            "Agent A": H_LABELS[m["a"]],
            "Agent B": H_LABELS[m["b"]],
            "W - D - L (A)": f"{s['wins']} - {s['draws']} - {s['losses']}",
            "Score A": round(s["score"], 4),
            "Bootstrap 95% CI": f"[{ci[0]:.3f}, {ci[1]:.3f}]",
        })

df_h_pairs = pd.DataFrame(h_pair_rows)
print("=== ĐỐI ĐẦU TRỰC TIẾP: BẢN ĐẦY ĐỦ vs CHỈ ĐẾM QUÂN (DISCS ONLY) ===")
display(df_h_pairs[df_h_pairs["Mode"] == "default_config"].reset_index(drop=True))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.0), dpi=140)
mode_data = s_heur["modes"]["default_config"]
order = ["alphabeta", "heuristic", "alphabeta_discs_only", "heuristic_discs_only"]
mat = np.full((len(order), len(order)), np.nan)
for i, ra in enumerate(order):
    for j, ca in enumerate(order):
        if i != j:
            mat[i, j] = mode_data["score_matrix"][ra][ca]

im = ax1.imshow(mat, cmap="RdYlGn", vmin=0.0, vmax=1.0)
ax1.set_xticks(range(len(order)), [H_LABELS[a] for a in order], rotation=22, ha="right", fontsize=9)
ax1.set_yticks(range(len(order)), [H_LABELS[a] for a in order], fontsize=9)
ax1.set_title("Ma trận đối đầu Ablation 3 (default_config)", fontweight="bold")
for i in range(len(order)):
    for j in range(len(order)):
        if i == j:
            ax1.text(j, i, "—", ha="center", va="center", fontweight="bold")
        else:
            val = mat[i, j]
            col = "white" if (val < 0.2 or val > 0.82) else "black"
            ax1.text(j, i, f"{val:.3f}", ha="center", va="center", color=col, fontweight="bold")

# Biểu đồ cột so sánh Mean Score của bản Full vs Discs-Only
x = np.arange(len(order))
scores_def = [s_heur["modes"]["default_config"]["mean_score"][a] for a in order]
scores_tpm = [s_heur["modes"]["time_per_move"]["mean_score"][a] for a in order]
w = 0.35
b1 = ax2.bar(x - w/2, scores_tpm, w, label="time_per_move (0.5s)", color="#2b6cb0")
b2 = ax2.bar(x + w/2, scores_def, w, label="default_config", color="#38a169")
for b in b2:
    ax2.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02, f"{b.get_height():.3f}", ha="center", fontsize=8.5, fontweight="bold")

ax2.set_xticks(x, [H_LABELS[a] for a in order], rotation=18, ha="right", fontsize=9)
ax2.set_ylim(0, 1.05)
ax2.axhline(0.5, color="gray", linestyle="--", lw=0.9)
ax2.set_ylabel("Mean Score (trung bình trước 3 biến thể còn lại)")
ax2.set_title("Sức mạnh: Đánh giá vị trí/cơ động vs Chỉ đếm quân", fontweight="bold")
ax2.legend()
ax2.grid(axis="y", linestyle=":", alpha=0.5)

fig.tight_layout()
fig.savefig(FIGS / "02_ablation_heuristic_components.png", bbox_inches="tight")
plt.show()
print("Đã lưu hình:", FIGS / "02_ablation_heuristic_components.png")